# Hybrid Recommender — the baseline, and your changes
**MKTG 4604 · Fall 2026 · Group build**

**Group members:** *(replace with all names)*

This notebook is a working recommender for a small bookseller. It was written by hand, by someone without
AI, and it is deliberately basic. **It runs as given: upload the three CSV files and Run all.** Your job is
not to make it run; it is to decide what is wrong with it, change it, and show whether your change helped.

How to use it:

- **Part 0** audits the data. **Parts 1–3** are the baseline. Read them; don't rewrite them.
- **Part 4** is where you add your invented readers and judge what the baseline gives them.
- **Part 5** is where you make your changes — one at a time — and judge again.
- Comments marked `# CHOICE:` are decisions the baseline made for you, each with a question. The menu of
  changes at Part 5 starts from these.

**If you don't code:** Google Colab → upload the three CSVs (folder icon, left) → *Runtime → Run all*.
Read every printed table. Fill in the `people` list in Part 4 (it is a list, not code). You own the audit,
the readers, the hand verdicts, the case for each change, and the judgment. Coders own Part 5.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import dendrogram, linkage

pd.set_option('display.width', 160)

In [ ]:
# The three files. ISBN is kept as text: some ISBNs end in 'X' and some start with 0.
book_data = pd.read_csv('book_data_sample_ed.csv',    dtype={'ISBN': str})
ratings   = pd.read_csv('ratings_data_sample_ed.csv', dtype={'ISBN': str})
users     = pd.read_csv('users_data_sample_ed.csv')

print('books', book_data.shape, '| ratings', ratings.shape, '| users', users.shape)

## Part 0 — What is in the data

Small organisation, data collected for other purposes, never cleaned. Read the output before you believe
anything the system says.

In [ ]:
print('--- ratings ---')
print('rows:', len(ratings), '| readers:', ratings['User-ID'].nunique(), '| books rated:', ratings['ISBN'].nunique())
print('\nratings per reader:');  print(ratings.groupby('User-ID').size().describe().round(2))
print('\nhow often each rating value appears:'); print(ratings['Book-Rating'].value_counts().sort_index())
print('\n--- users ---')
print('rows:', len(users), '| age missing:', users['Age'].isna().sum())
print('age:'); print(users['Age'].describe().round(1))
print('\n--- books ---')
print('rows:', len(book_data), '| distinct ISBN:', book_data['ISBN'].nunique(), '| distinct titles:', book_data['title'].nunique())
print('\nmost-rated books:')
top5 = ratings['ISBN'].value_counts().head(5).rename('n_ratings').reset_index().rename(columns={'index': 'ISBN'})
print(top5.merge(book_data[['ISBN', 'title']].drop_duplicates('ISBN'), on='ISBN')[['title', 'n_ratings']].to_string(index=False))

**Audit (Paragraph A of your PDF starts here).** What is this data, and what does it mean for a system that
learns from "readers like you"? How much does a typical reader tell it? What is a rating of 0? What is wrong
with the ages? Is anything in the book file duplicated or mislabelled?

## Part 1 — Baseline: group the readers (collaborative filtering)

In [ ]:
# One row per reader, one column per book, the rating in the cell.
# CHOICE: empty cells are filled with 0. The file also records 0 for "read, not rated".
#   Q: Should "never saw it" and "read it, didn't rate it" be the same number? What does that do to a
#      book's average?
def build_matrix(ratings, users):
    user_item = ratings.pivot_table(index='User-ID', columns='ISBN', values='Book-Rating', fill_value=0)
    ages = pd.to_numeric(users.set_index('User-ID').loc[user_item.index, 'Age'], errors='coerce')
    # CHOICE: missing ages are filled with the median age. Ages of 1 and 239 are kept.
    #   Q: Are those readers? What does keeping them do to the groups?
    ages = ages.fillna(ages.median())
    return user_item, ages

user_item, ages = build_matrix(ratings, users)

# Age is scaled (mean 0, spread 1) and appended as the last column, so readers are grouped on ratings AND age.
# CHOICE: age is ONE column beside the book columns; AGE_WEIGHT multiplies it.
#   Q: With about a hundred columns of ratings and one of age, which decides who lands together?
#      What should "readers like you" mean for this bookseller?
AGE_WEIGHT = 1.0

age_scaler = StandardScaler().fit(ages.to_frame())
features   = np.hstack([user_item.values, age_scaler.transform(ages.to_frame()) * AGE_WEIGHT])
n_items    = user_item.shape[1]
print('feature matrix:', features.shape, '= readers x (books + age)')

In [ ]:
# How many groups? Hierarchical clustering on a sample of 500 readers, read off the dendrogram.
sample = pd.DataFrame(features, index=user_item.index).sample(n=500, random_state=42)
plt.figure(figsize=(12, 4)); dendrogram(linkage(sample.values, method='ward'), no_labels=True)
plt.title('500 readers, grouped by ratings and age'); plt.ylabel('distance'); plt.show()

# CHOICE: K = 3 groups.
#   Q: Is that what the dendrogram shows? Could the bookseller act on more? On fewer?
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10).fit(features)
reader_clusters = pd.DataFrame({'User-ID': user_item.index, 'Cluster': kmeans.labels_, 'Age': ages.values})

print(reader_clusters['Cluster'].value_counts().sort_index().rename('readers'))
print(reader_clusters.groupby('Cluster')['Age'].agg(['mean', 'min', 'max']).round(1))
for c in range(K):
    ids = reader_clusters.loc[reader_clusters.Cluster == c, 'User-ID']
    sub = ratings[ratings['User-ID'].isin(ids)]
    top = (sub.groupby('ISBN').agg(n=('User-ID', 'count'), mean_rating=('Book-Rating', 'mean'))
              .sort_values('n', ascending=False).head(4)
              .merge(book_data[['ISBN', 'title']].drop_duplicates('ISBN'), on='ISBN', how='left'))
    print(f'\n=== Group {c}: {len(ids)} readers, {len(sub)} ratings ===')
    print(top[['title', 'n', 'mean_rating']].round(2).to_string(index=False))

**For Paragraph A:** describe each group in one sentence a marketing manager would use. What are the
groups really separating on — taste, age, or one popular book?

**Draft for Paragraph A (from the Part 1 output above, before our test readers are added)**

*The three groups, in marketing-manager terms*

- **Group 0, "Poisonwood Bible fans" (34 readers, 3%):** a small niche where every member rated *The Poisonwood Bible*, and rated it highly (8.3/10 on average). They've said almost nothing about any other book.
- **Group 1, "the silent majority" (786 readers, 75%):** casual readers who typically rated about one book, usually with a 0 ("read it, didn't rate it"). They gave *The Five People You Meet in Heaven* an average of 0.1 and *The Poisonwood Bible* 0.0. This is a leftover bucket, not a segment.
- **Group 2, "Five People You Meet in Heaven fans" (232 readers, 22%):** every member rated *The Five People You Meet in Heaven*, and rated it highly (8.2/10). Beyond that one book, we know almost nothing about their tastes.

*What the groups are really separating on*

- **Not age:** average ages are almost the same (38, 35 and 40), and each group covers nearly the full range, from teens to impossible ages like 116 and 239. Strangers aged 19, 30, 45 and 60 all land in Group 1.
- **Not taste in any real sense:** most readers rated only one or two books, so there's no pattern of preferences to group on.
- **One popular book, really two:** *The Five People You Meet in Heaven* accounts for 427 of about 1,200 ratings. The groups come down to whether you gave that book (or *The Poisonwood Bible*) a high score, versus a 0 or nothing.
- **The 0 problem makes it worse:** "never read it" and "read it, didn't rate it" are both stored as 0. So Group 1 is mostly readers with zeros, not readers who disliked these books.
- **What it means for the bookseller:** "readers like you" currently means "readers who scored the same bestseller the same way." That's too thin to drive recommendations.

*Note:* re-check these numbers against the printed tables if the data or settings change. Group numbers and sizes also shift after Part 4 adds the test readers (9 of our 10 land in the same group).


## Part 2 — Baseline: from a group to a candidate list

In [ ]:
# Which group does a reader belong to?
#   A KNOWN reader (in the data): read it off reader_clusters.
#   A STRANGER (age only): build a row with every book column 0 and the age in the age column, and ask k-means.
#   Q: Try several ages for a stranger. Does the group change? What does the baseline know about a new reader?
def group_of(age=None, user_id=None):
    if user_id is not None:
        return int(reader_clusters.loc[reader_clusters['User-ID'] == user_id, 'Cluster'].iloc[0])
    vec = np.zeros(n_items + 1)
    vec[n_items] = age_scaler.transform([[age]])[0, 0] * AGE_WEIGHT
    return int(kmeans.predict([vec])[0])

for a in [19, 30, 45, 60]:
    print(f'stranger aged {a} -> group {group_of(age=a)}')

In [ ]:
# The CANDIDATE list: the 25 books the reader's group rated highest.
# CHOICE: ranked by mean rating, ties broken by how many readers rated the book. Zeros count as ratings.
#   Q: Look at rating_count for the top five. How many readers is each "best book" based on?
#      Would you trust a restaurant with one five-star review over one with four hundred four-star reviews?
def candidate_list(group, n=25):
    ids = reader_clusters.loc[reader_clusters.Cluster == group, 'User-ID']
    sub = ratings[ratings['User-ID'].isin(ids)]
    agg = (sub.groupby('ISBN').agg(mean_rating=('Book-Rating', 'mean'), rating_count=('User-ID', 'count'))
              .reset_index()
              .sort_values(['mean_rating', 'rating_count'], ascending=False)
              .head(n))
    return agg.merge(book_data.drop_duplicates('ISBN'), on='ISBN', how='left')   # one ISBN is duplicated in the book file

candidate_list(group_of(age=30))[['title', 'mean_rating', 'rating_count']].head(10)

## Part 3 — Baseline: re-rank the candidates by the book the reader liked (content-based filtering)

Every book becomes a vector: its genres (one column each), the words of its description (TF-IDF), its
average rating and year (scaled 0–1). Two books are similar when their vectors point the same way
(cosine similarity). The 25 candidates are sorted by similarity to the one book the reader said they liked.

In [ ]:
books = book_data.drop_duplicates('ISBN').reset_index(drop=True)
books['genre_list'] = books['genres'].apply(ast.literal_eval)

genre_matrix = MultiLabelBinarizer().fit_transform(books['genre_list'])
desc_matrix  = TfidfVectorizer(stop_words='english').fit_transform(books['description'].fillna('')).toarray()
num_matrix   = MinMaxScaler().fit_transform(books[['av_rating', 'Year-Of-Publication']].fillna(0))

# CHOICE: the three blocks are stacked with equal weight. Genres: ~30 columns. Description words: ~3,000. Numbers: 2.
#   Q: Which block do you think actually decides similarity here? How would you check?
book_vectors = np.hstack([genre_matrix, desc_matrix, num_matrix])
sim_df = pd.DataFrame(cosine_similarity(book_vectors), index=books['title'], columns=books['title'])

def similarity_to(liked_book):
    s = sim_df[liked_book].drop(index=liked_book).reset_index()
    s.columns = ['title', 'similarity']
    return s

similarity_to('the hobbit').sort_values('similarity', ascending=False).head(5)

In [ ]:
# THE RECOMMENDER. One function, two ways to call it:
#   recommend('the hobbit', age=19)         a stranger: placed in a group by age alone
#   recommend('the hobbit', user_id=9001)   a known reader: served from their own group
# Every call is logged (Part 4).
recommendation_log = []

def recommend(liked_book, age=None, user_id=None, n=10):
    group = group_of(age=age, user_id=user_id)
    cands = candidate_list(group)
    out   = (cands.merge(similarity_to(liked_book), on='title', how='inner')
                  .sort_values('similarity', ascending=False).head(n).copy())
    # CHOICE: the reason is a template sentence.  Q: Would a reader believe it? What would a better one say?
    genres = out['title'].map(books.set_index('title')['genre_list'])
    out['reason'] = genres.apply(lambda g: f"readers in your group rated it well; shares {g[0].lower() if isinstance(g, list) and g else 'a genre'} with your pick")
    recommendation_log.append({'when': pd.Timestamp.now(), 'liked_book': liked_book, 'age': age, 'user_id': user_id,
                               'group': group, 'shown': list(out['title']), 'version': MODEL_VERSION})
    return out[['title', 'Book-Author', 'mean_rating', 'rating_count', 'similarity', 'reason']].reset_index(drop=True)

MODEL_VERSION = 'baseline'
recommend('the hobbit', age=19)

## Part 4 — Your readers, and the baseline's verdict (Paragraph A ends here; Paragraph B starts)

Your group invented ten readers on the individual pages. Add them all here. **They are test readers, not
new data** — ten people with three ratings each will not change what the model learned from a thousand.
They are how you find out what it does.

Fill the list below. **Titles must be copied exactly from the `title` column of the book file**
(lower case, as printed there). Ratings 1–10. Keep the one rated highest as the book they "liked".

In [ ]:
# PLACEHOLDER READERS (invented for testing) - replace with the group's real readers from the individual specs.
# 'trap' = a reader built to expose a specific weakness of the baseline.
people = [
    # --- typical readers ---
    {'name': 'Priya',   'age': 19, 'trap': False, 'books': [('the hobbit', 9), ('the fellowship of the ring (the lord of the rings, part 1)', 8), ("ender's game", 7)]},
    {'name': 'Daniel',  'age': 22, 'trap': False, 'books': [('1984', 10), ('brave new world', 8), ('fahrenheit 451', 7)]},
    {'name': 'Elena',   'age': 34, 'trap': False, 'books': [('pride and prejudice', 10), ('jane eyre', 9), ('wuthering heights', 7)]},
    {'name': 'Viktor',  'age': 47, 'trap': False, 'books': [('crime and punishment', 9), ('the brothers karamazov', 10), ('war and peace', 8)]},
    {'name': 'Grace',   'age': 41, 'trap': False, 'books': [('the kite runner', 9), ('memoirs of a geisha', 8), ('a tree grows in brooklyn', 8)]},
    # --- traps ---
    # Grandparent buying picture books: age says "adult classics", taste says "children's".
    {'name': 'Marcus',  'age': 68, 'trap': True,  'books': [('green eggs and ham (i can read it all by myself beginner books)', 10), ('the cat in the hat', 9), ('goodnight moon', 9)]},
    # Science reader: 'a brief history of time' is labelled Historical Fiction in the book file.
    {'name': 'Sam',     'age': 29, 'trap': True,  'books': [('a brief history of time', 10), ("the hitchhiker's guide to the galaxy", 6), ('frankenstein', 5)]},
    # Harry Potter fan: 'goblet of fire' appears twice in the catalogue under two titles.
    {'name': 'Leo',     'age': 13, 'trap': True,  'books': [('harry potter and the goblet of fire', 10), ('harry potter and the chamber of secrets (book 2)', 9), ("harry potter and the sorcerer's stone (harry potter (hardcover))", 9)]},
    # Only knows the bestseller everyone rated: does the system just echo the crowd?
    {'name': 'Linda',   'age': 55, 'trap': True,  'books': [('the five people you meet in heaven', 10), ('the poisonwood bible', 4), ('little house on the prairie', 6)]},
    # Picky reader: their "highest" book is only a 5, so the "liked" book isn't really liked.
    {'name': 'Omar',    'age': 36, 'trap': True,  'books': [('atlas shrugged', 5), ('the alchemist', 2), ('the stranger', 3)]},
]

def add_people(people, start_id=9001):
    """Add invented readers to users and ratings. Looks each title up in the catalogue; reports what it found."""
    global ratings, users
    title_to_isbn = books.set_index('title')['ISBN'].to_dict()
    new_users, new_ratings, ids = [], [], {}
    for i, p in enumerate(people):
        uid = start_id + i; ids[p['name']] = uid
        new_users.append({'User-ID': uid, 'Location': 'test reader', 'Age': p['age']})
        found = 0
        for title, r in p['books']:
            if title in title_to_isbn:
                new_ratings.append({'User-ID': uid, 'ISBN': title_to_isbn[title], 'Book-Rating': r}); found += 1
            else:
                print(f"  !! {p['name']}: '{title}' is not in the catalogue — copy the title exactly from the book file")
        print(f"{p['name']} added as user {uid}: {found} of {len(p['books'])} titles found")
    users   = pd.concat([users[~users['User-ID'].isin(ids.values())],     pd.DataFrame(new_users)],   ignore_index=True)
    ratings = pd.concat([ratings[~ratings['User-ID'].isin(ids.values())], pd.DataFrame(new_ratings)], ignore_index=True)
    return ids

if people:
    person_ids = add_people(people)
    # Re-run Parts 1–2 so your readers have groups.  (Everything the baseline does, in one call.)
    user_item, ages = build_matrix(ratings, users)
    age_scaler = StandardScaler().fit(ages.to_frame())
    features   = np.hstack([user_item.values, age_scaler.transform(ages.to_frame()) * AGE_WEIGHT])
    n_items    = user_item.shape[1]
    kmeans     = KMeans(n_clusters=K, random_state=42, n_init=10).fit(features)
    reader_clusters = pd.DataFrame({'User-ID': user_item.index, 'Cluster': kmeans.labels_, 'Age': ages.values})
    for name, uid in person_ids.items():
        print(f'{name}: group {group_of(user_id=uid)}')

**Judge the baseline.** Pick five of your ten readers (one per member; at least two of them traps). For each,
run the two lines below, print the ten, and give every recommendation one letter: **S** sensible for this
person · **O** obvious (they would have found it anyway, or already have it) · **W** wrong. Fifty letters.
Write the five lines of the verdict table in your PDF; the counts go in the ladder.

In [21]:
# The five readers we judge (one per member, at least two traps). Change the names if the group picks others.
JUDGE = ['Priya', 'Daniel', 'Marcus', 'Sam', 'Leo']

def distinct_titles(names):
    """How many different books appear across these readers' latest lists (out of 10 x readers)."""
    shown = [row['shown'] for row in recommendation_log if row['user_id'] in [person_ids[n] for n in names] and row['version'] == MODEL_VERSION]
    return len({t for lst in shown for t in lst}), 10 * len(shown)

def judge_readers(names=JUDGE):
    """Run each reader as a known reader (and as a stranger, for comparison), print the lists,
    and save a grading sheet: one row per recommendation, with a blank 'verdict' column for S / O / W."""
    by_name = {p['name']: p for p in people}
    rows = []
    for name in names:
        p = by_name[name]
        liked = max(p['books'], key=lambda b: b[1])[0]           # the book they rated highest
        known    = recommend(liked, user_id=person_ids[name])
        stranger = set(recommend(liked, age=p['age'])['title'])  # same person, age only
        rated = '; '.join(f'{t} ({r})' for t, r in p['books'])
        print(f"\n=== {name}, {p['age']}{' (TRAP)' if p.get('trap') else ''} | liked: {liked} ===")
        print(known[['title', 'mean_rating', 'rating_count', 'similarity']].round(2).to_string())
        print(f'same as stranger list: {len(set(known["title"]) & stranger)} of {len(known)}')
        for rank, rec in known.iterrows():
            rows.append({'version': MODEL_VERSION, 'reader': name, 'age': p['age'], 'trap': p.get('trap', False),
                         'rated_books': rated, 'liked_book': liked, 'rank': rank + 1,
                         'recommended': rec['title'], 'author': rec['Book-Author'],
                         'genres': ', '.join(books.set_index('title')['genre_list'].get(rec['title'], [])[:4]),
                         'already_rated_by_reader': rec['title'] in {t for t, _ in p['books']},
                         'also_in_stranger_list': rec['title'] in stranger,
                         'verdict': '', 'why': ''})
    sheet = pd.DataFrame(rows)
    fname = 'grading_' + ''.join(ch if ch.isalnum() else '_' for ch in MODEL_VERSION) + '.csv'
    sheet.to_csv(fname, index=False)
    d, total = distinct_titles(names)
    print(f'\ndistinct titles: {d} of {total}')
    print(f'saved {fname}: {len(sheet)} rows. In Colab: folder icon on the left -> right-click the file -> Download.')
    return sheet

if people:
    baseline_sheet = judge_readers()



=== Priya, 19 | liked: the hobbit ===
                                                                           title  mean_rating  rating_count  similarity
0                                               alice's adventures in wonderland         5.67             3        0.68
1                             harry potter and the goblet of fire (harry potter)        10.00             1        0.58
2                                                              the little prince         6.18            11        0.58
3                                           the hitchhiker's guide to the galaxy         7.50             2        0.58
4                                                      oh, the places you'll go!        10.00             2        0.57
5                                            harry potter and the goblet of fire        10.00             1        0.48
6  where the sidewalk ends 30th anniversary special edition : poems and drawings        10.00             1        0.48
7

/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## Part 5 — Your changes (Paragraphs B and C)

**The rule: change one thing, re-run your five readers, judge, keep or throw away. Then the next.** Three
changes at once with one before/after tells you nothing about which one worked.

**The menu** (start anywhere; add your own):
1. **Swap the order** — content-based first across the whole catalogue, then the group's ratings to re-rank.
2. **Fix the ranking** — a minimum rating count, or a Bayesian average that shrinks a book's mean toward the global mean when few people rated it.
3. **Treat 0 as "read, unrated"** — leave it out of the means.
4. **Age bands instead of k-means** (<25, 25–45, 45+) — did the clustering earn its keep?
5. **Raise `AGE_WEIGHT`** — or lower it to zero.
6. **Item-based** — books that people who liked *this* book were unusually likely to also like.
7. **A diversity cap** — no more than three books from one genre in the ten.
8. **Better reasons** — a real sentence per book, written by rule or by an AI model from the titles and genres.
9. **Treat strangers differently** — the baseline places a new reader by age alone; what else could it use?

For each change: set `MODEL_VERSION` to a short name, rewrite the function it touches (copy it below, don't
edit the baseline), re-run your five readers, count S/O/W and distinct titles, decide. A change you tried and
**rejected** counts in your favour in the trace if you say why.

In [ ]:
# ---- Change 1 ----
# WHAT: never recommend a book the reader has already rated, and never show the same book twice
#       under two titles (e.g. 'harry potter and the goblet of fire' vs '... (harry potter)').
# WHY:  in the baseline, Marcus and Daniel were shown books they had already rated (automatic O's),
#       and Leo's #1 was the book he liked, under its other title.
# TOUCHES: recommend() only. Groups, candidates and similarity are unchanged.
MODEL_VERSION = 'change-1: drop already-rated and duplicate titles'

import re
def base_title(t):
    """'harry potter and the goblet of fire (harry potter)' -> 'harry potter and the goblet of fire'."""
    return re.sub(r'\s*\(.*$', '', str(t)).strip()

def recommend(liked_book, age=None, user_id=None, n=10):
    group = group_of(age=age, user_id=user_id)
    cands = candidate_list(group)
    out   = (cands.merge(similarity_to(liked_book), on='title', how='inner')
                  .sort_values('similarity', ascending=False).copy())
    # NEW: remove anything the reader already rated (matched on base title, so duplicates count too)
    seen = {base_title(liked_book)}
    if user_id is not None:
        rated_isbns = ratings.loc[ratings['User-ID'] == user_id, 'ISBN']
        seen |= set(books.loc[books['ISBN'].isin(rated_isbns), 'title'].map(base_title))
    out = out[~out['title'].map(base_title).isin(seen)]
    # NEW: if two titles are the same book, keep only the first (most similar) one
    out = out[~out['title'].map(base_title).duplicated()].head(n).copy()
    genres = out['title'].map(books.set_index('title')['genre_list'])
    out['reason'] = genres.apply(lambda g: f"readers in your group rated it well; shares {g[0].lower() if isinstance(g, list) and g else 'a genre'} with your pick")
    recommendation_log.append({'when': pd.Timestamp.now(), 'liked_book': liked_book, 'age': age, 'user_id': user_id,
                               'group': group, 'shown': list(out['title']), 'version': MODEL_VERSION})
    return out[['title', 'Book-Author', 'mean_rating', 'rating_count', 'similarity', 'reason']].reset_index(drop=True)

# Re-run the same five readers and save a new grading sheet.
if people:
    change1_sheet = judge_readers()
    print('\nalready-rated books still shown:', int(change1_sheet['already_rated_by_reader'].sum()))


In [ ]:
# ---- Change 2 ----
# WHAT: rank the group's candidates by a Bayesian average instead of the raw mean.
#       Each book's mean is pulled toward the overall average rating; the fewer people rated it,
#       the harder it is pulled. A book needs many good ratings to reach the top, not one lucky 10.
#         score = (n * book_mean + M * overall_mean) / (n + M)
# WHY:  the baseline's "best books" were 10/10 from one or two readers (look at rating_count),
#       so the 25 candidates were mostly noise, and Change 1's replacements came from that noise.
# TOUCHES: candidate_list() only. Keeps Change 1's recommend(). Zeros still count (that's a separate change).
MODEL_VERSION = 'change-2: bayesian average ranking'

M = 5   # how many "average" ratings each book is assumed to start with. Higher = more trust in popular books.

def candidate_list(group, n=25):
    ids = reader_clusters.loc[reader_clusters.Cluster == group, 'User-ID']
    sub = ratings[ratings['User-ID'].isin(ids)]
    overall_mean = sub['Book-Rating'].mean()
    agg = (sub.groupby('ISBN').agg(mean_rating=('Book-Rating', 'mean'), rating_count=('User-ID', 'count'))
              .reset_index())
    agg['score'] = (agg['rating_count'] * agg['mean_rating'] + M * overall_mean) / (agg['rating_count'] + M)
    agg = agg.sort_values(['score', 'rating_count'], ascending=False).head(n)
    return agg.merge(book_data.drop_duplicates('ISBN'), on='ISBN', how='left')

# Re-run the same five readers and save a new grading sheet.
if people:
    change2_sheet = judge_readers()


In [ ]:
# ---- Change 3 ----
# WHAT: treat a rating of 0 as "read, not rated" and leave it out of the averages.
#       A book's mean and count now come only from real scores (1-10). Books with no real scores drop out.
# WHY:  the file records 0 for "read, didn't rate it", so a zero is not a bad review. Counting zeros
#       dragged down widely-read books (e.g. The Giving Tree averaged 5.6 over 33 ratings).
# TOUCHES: candidate_list() only. Keeps Change 1's recommend() and Change 2's Bayesian average.
#          The clustering (Part 1) still fills blanks with 0 - fixing that is a separate change.
MODEL_VERSION = 'change-3: ignore zero ratings'

def candidate_list(group, n=25):
    ids = reader_clusters.loc[reader_clusters.Cluster == group, 'User-ID']
    sub = ratings[ratings['User-ID'].isin(ids) & (ratings['Book-Rating'] > 0)]   # NEW: real ratings only
    overall_mean = sub['Book-Rating'].mean()
    agg = (sub.groupby('ISBN').agg(mean_rating=('Book-Rating', 'mean'), rating_count=('User-ID', 'count'))
              .reset_index())
    agg['score'] = (agg['rating_count'] * agg['mean_rating'] + M * overall_mean) / (agg['rating_count'] + M)
    agg = agg.sort_values(['score', 'rating_count'], ascending=False).head(n)
    return agg.merge(book_data.drop_duplicates('ISBN'), on='ISBN', how='left')

# Re-run the same five readers and save a new grading sheet.
if people:
    change3_sheet = judge_readers()


## Before and after

For the PDF: the **ladder** (one row per change you kept: name · S/O/W counts across the fifty · distinct
titles of 50) and the **before/after table** (five readers × ten titles, baseline beside final). The cell
below prints the log so you can build both from it.

In [ ]:
log = pd.DataFrame(recommendation_log)
log[['version', 'user_id', 'age', 'liked_book', 'shown']] if len(log) else print('no recommendations logged yet')

## Paragraph C — Judgment, and the trace

Did your changes help, by your fifty letters and your distinct-titles count? What can this test *not* tell
you — and what would the bookseller have to collect to find out? (It can only reward books your five readers
would already have liked. What kind of good recommendation can it never give credit for?)

**The trace** goes at the end of your PDF: 3–6 rows — *what we asked the AI tool · what it gave us · what we
checked · what we changed* — including changes you tried and threw away. You may ask the tool to help write
it. You may not let it be the only one who read it.

**Draft for Paragraph C (fill the [brackets] once the grading sheets are back)**

*Did our changes help?*

- **By our fifty letters:** baseline [__ S / __ O / __ W] → after Change 1 [__ / __ / __] → Change 2 [__ / __ / __] → Change 3 [__ / __ / __]. [One sentence: which change moved the S count the most.]
- **By distinct titles:** no. Variety stayed flat: 23 → 20 → 22 → 21 of 50. Our changes improved *which* books were picked, not *how many different* books readers saw.
- **Change 1 (drop already-rated and duplicates)** did exactly what it was meant to: books the reader already rated fell from 4 to 0, and Leo no longer gets his own favourite back under a second title. But the empty slots were filled from a weak pool (Leo, 13, got Stephen King's *The Stand*).
- **Change 2 (Bayesian average)** helped most where there was real data: Priya now gets *The Lord of the Rings* first, and Daniel's list became adult fiction instead of children's poetry.
- **Change 3 (ignore zeros)** fixed the biggest data error (63% of ratings in our readers' group were 0 = "read, not rated"), but it partly undid Change 2: without zeros nearly every rating is high, so books with a single 10 crept back into the top 25. Two sensible fixes can cancel each other out.
- **What none of our changes fixed:** every reader got the *same* ten books as a known reader and as a stranger (10 of 10), and 9 of our 10 readers fell in one group. The "readers like you" step is still not personalising anything. Sam's list stayed wrong because his favourite book is mislabelled as Historical Fiction; no ranking change can fix bad catalogue data.
- [Decision: which changes we keep, and why. A rejected change counts in our favour if we say why.]

*What this test cannot tell us*

- It can only reward books our readers **would already have liked**. Our readers are invented by us, with tastes we chose, so a "sensible" letter means "matches what we assumed." It can never give credit for **discovery**: a book a reader would never have guessed but would love. That is the recommendation a bookseller most wants, and our test is blind to it.
- Five readers and fifty letters, graded by the same people who made the changes and who knew which version they were looking at. That is a small, biased sample.
- It says nothing about whether anyone would **buy**, click, or come back.

*What the bookseller would need to collect to find out*

- What people **did** after seeing a recommendation: clicked, bought, finished, rated it afterwards. Ideally an A/B test: half the customers get the old list, half the new one.
- A clear difference between **"never saw it"** and **"read it, didn't rate it"** (today both are stored as 0).
- More ratings per reader (most rated one book), a way to say "not interested", and a **checked catalogue**: correct genres, no duplicate titles, no impossible ages.

*The trace (what we asked the AI tool · what it gave us · what we checked · what we changed)*

| What we asked | What it gave us | What we checked | What we changed |
|---|---|---|---|
| Explain the project and the baseline | A walkthrough of the three steps (group → candidates → re-rank by similarity) and the main flaws | Ran the notebook; confirmed the numbers (427 of ~1,200 ratings on one book, ages 1 and 239, strangers aged 19–60 all in one group) | [ ] |
| Draft Paragraph A bullets | Group descriptions and what the groups separate on | Compared every number to the Part 1 printout | [e.g. rewrote in our own words] |
| Invent ten placeholder test readers, including traps | 5 typical readers + 5 traps (age mismatch, mislabelled book, duplicate title, bestseller-only, picky reader) | All 30 titles found in the catalogue; traps exposed the problems they targeted | [replaced with our own readers from the individual specs?] |
| Code a grading sheet + Change 1 (drop already-rated and duplicates) | `judge_readers()` and a new `recommend()` | Already-rated books shown fell from 4 to 0; distinct titles 23 → 20 | [kept / rejected, why] |
| Code Change 2 (Bayesian average) and Change 3 (ignore zeros) | New `candidate_list()` versions | Top candidates now have more ratings behind them; found 63% zeros; noticed single-vote 10s returned after Change 3 | [kept / rejected, why] |

*Rule from the brief: the AI may help write the trace, but it can't be the only one who read it. Each of us should check at least one row.*
